# inplace-param-update — worked example 1: In-place momentum SGD step preserves storage

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-param-update`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A hand-rolled optimizer must update parameters IN PLACE (`p.data -= ...` or `p.data.add_(...)`), never by rebinding (`p = p - ...`). The model and the optimizer both hold the original `nn.Parameter` object; rebinding a local name leaves their copy untouched, so the model never learns. In-place mutation changes the storage the model already points at, keeping its `data_ptr()` stable.

## Worked solution

We implement one step of SGD-with-momentum over a parameter list and prove the storage is preserved.

1. **Velocity buffer.** Momentum keeps a running velocity per parameter: `v = mu * v + g`. We store these in a list initialized to zeros so the first step is plain gradient descent.
2. **Update the buffer in place.** `v.mul_(mu).add_(g)` mutates the velocity tensor without allocating a new one — clean and allocation-free.
3. **Update the parameter in place.** `p.data.add_(v, alpha=-lr)` does `p.data -= lr * v`. Using `.data` bypasses autograd so this write is not tracked, and `add_` keeps the same storage that the model already references.
4. **Why not rebind.** `p = p - lr * v` would make the local name `p` point at a fresh tensor and silently drop the update. We avoid it entirely.

The demo records each parameter's `data_ptr()` before and after the step and confirms every pointer is unchanged while the values moved.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)

def momentum_step(params, grads, velocities, lr, mu):
    for p, g, v in zip(params, grads, velocities):
        v.mul_(mu).add_(g)            # v = mu*v + g, in place
        p.data.add_(v, alpha=-lr)     # p -= lr * v, in place (no rebind)
    return params

lin = nn.Linear(3, 2)
params = [lin.weight, lin.bias]
grads = [t.ones_like(p) for p in params]
vels = [t.zeros_like(p) for p in params]

ptrs_before = [p.data_ptr() for p in params]
vals_before = [p.detach().clone() for p in params]
momentum_step(params, grads, vels, lr=0.1, mu=0.9)
ptrs_after = [p.data_ptr() for p in params]

print('storage preserved:', ptrs_before == ptrs_after)
print('weight changed:', not t.allclose(vals_before[0], params[0]))